# 01.2 — When you don't need RAG

A developer once spent two weeks building a full RAG pipeline for an internal
documentation site. Vector database, chunking strategy, embedding model
selection, retrieval evaluation, deployment. The entire corpus was about 150,000
tokens. It would have fitted in a single prompt.

Before you spend a week on the rest of this course, spend twenty minutes finding
out whether you need it.

In [1]:
!pip install -q pymupdf4llm==1.28.2 openai==3.8.0

## Measure your corpus first

Not "is it big?" — actually count it.

In [2]:
from pathlib import Path
import pymupdf4llm

CORPUS = Path('../../corpus/docs')

NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]

docs = {name: pymupdf4llm.to_markdown(str(CORPUS / name)) for name in NATIVE_PDFS}

for name, text in docs.items():
    print(f'{len(text):>7,}  {name}')

everything = '\n\n'.join(f'=== {n} ===\n{t}' for n, t in docs.items())
print(f'\n{len(everything):>7,}  TOTAL characters')
print(f'{len(everything) // 4:>7,}  tokens, roughly (about 4 characters each)')

  6,472  sahel-employee-handbook-2023.pdf
  6,559  sahel-employee-handbook-2025.pdf
  3,595  sahel-procurement-policy-v3.pdf
  4,286  nfsc-circular-2024-07-cybersecurity.pdf
  2,300  nfsc-circular-2025-02-amendment.pdf
  6,120  kaduna-agro-annual-report-2024.pdf
  4,057  kaduna-agro-board-minutes-2024-10-17.pdf

 33,707  TOTAL characters
  8,426  tokens, roughly (about 4 characters each)


Around 8,000 tokens.

For context, current models accept far more than that. Claude Sonnet takes a
million tokens. Gemini takes two million. Llama 4 Scout advertises ten million.
Even a modest 128,000-token window holds this corpus fifteen times over.

So the honest question is not *how do I build retrieval*. It's *why would I*.

## Just put the whole thing in the prompt

No chunking, no embeddings, no vector database. Paste all seven documents in and
ask.

In [3]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url='https://openrouter.ai/api/v1',
)
CHAT_MODEL = 'minimax/minimax-m2.7:free' # Check https://openrouter.ai/models for free models

question = 'What is the maximum value of an emergency procurement without competitive sourcing?'

response = client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0,
    messages=[{
        'role': 'user',
        'content': f'Answer using only the documents below.\n\n{everything}\n\nQUESTION: {question}',
    }],
)

print(response.choices[0].message.content)
print()
print('prompt tokens:', response.usage.prompt_tokens)

Based on the **Sahel Microfinance Bank Plc Procurement and Vendor Management Policy (Version 3.1, effective 1 May 2024)**, the maximum value of an emergency procurement without competitive sourcing is **NGN 5,000,000**.

This is outlined in Section 4 (Emergency Procurement), which states:

> "Where a failure of a critical service would result in branch closure, loss of connectivity to the core banking application, or breach of a regulatory deadline, the Head of Administration may authorise emergency procurement up to NGN 5,000,000 without competitive sourcing."

This emergency provision must be reported to the Management Procurement Committee at its next meeting with written justification, and it may not be used for recurring requirements that could have been reasonably anticipated.

prompt tokens: 7334


It works. It is also considerably simpler than what you'll build in module 02.

`prompt_tokens` is the real number, straight from the API — not an estimate.
Write it down.

## So why build retrieval at all?

Four reasons, and the first one is arithmetic.

In [4]:
PROMPT_TOKENS = response.usage.prompt_tokens

# Look up your model's input price at https://openrouter.ai/models and put it here.
PRICE_PER_MTOK = 0.24   # USD per million input tokens — CHANGE THIS

RETRIEVED_TOKENS = 2_000   # roughly what a 5-chunk RAG prompt costs

for queries_per_day in (100, 10_000):
    stuffed = PROMPT_TOKENS * queries_per_day * 30 / 1e6 * PRICE_PER_MTOK
    rag = RETRIEVED_TOKENS * queries_per_day * 30 / 1e6 * PRICE_PER_MTOK
    print(f'{queries_per_day:>6,} queries/day  |  stuff: ${stuffed:>8,.2f}/mo   rag: ${rag:>7,.2f}/mo')

   100 queries/day  |  stuff: $    5.28/mo   rag: $   1.44/mo
10,000 queries/day  |  stuff: $  528.05/mo   rag: $ 144.00/mo


**Cost.** Stuffing means every question pays for every document, whether or not
it's relevant. At low volume that's cents. Scale it and the ratio — not the
absolute figure — is what bites.

**Latency.** A model reads the whole prompt before writing a word. Nvidia's
benchmark on a 70B model shows time-to-first-token rising from 31ms at 200 tokens
to 1,833ms at 10,000. Fifty times the tokens, about fifty-nine times the wait.

**Accuracy actually falls.** This is the counterintuitive one. Models are
measurably worse at finding information buried in the middle of a long context
than at either end — accuracy drops by 10 to 20 percentage points. A bigger
window is not a better window.

**It doesn't scale.** This is the real reason. Our corpus is seven documents.
Real ones are ten thousand. There is no context window that holds a company's
document store, and there never will be, because the store grows faster than the
window does.

## The honest decision rule

**Don't build RAG if:**

- Your corpus fits comfortably in a context window and isn't growing fast. Paste
  it in. You can always build retrieval later; you can't get the week back.
- Your data is structured. "How many vendors have expired tax certificates?" is a
  SQL query. Embedding a spreadsheet to answer it is worse in every way.
- You want to change *how* the model writes rather than *what it knows*. That's
  fine-tuning.
- The answer isn't in documents at all — it's in a live system. That's a tool
  call, or an agent.
- Exact keyword matching is what users actually do. A search box over an inverted
  index is cheaper, faster and easier to debug than anything in this course.

**Build RAG if:** the corpus is too large to stuff, changes often enough that
retraining is impractical, needs per-user access control, or has to cite its
sources.

Most real problems land in the second list. Enough land in the first that the
question is worth asking out loud.

## And when not to build it yourself

Separate question, equally worth asking. You can buy this.

There are end-to-end platforms — Vectara, Nuclia, Ragie, LlamaCloud — and there
is a middle tier from the cloud providers: Amazon Bedrock Knowledge Bases, Google
Vertex AI Search, Azure AI Search. Upload documents, get an endpoint.

For a straightforward internal Q&A tool, that is often the right answer, and a
course that pretended otherwise would be selling you something.

Four situations where it isn't:

**You can't debug what you don't understand.** When a platform returns a wrong
answer, you have a support ticket. When your own pipeline does, you have a
measurement problem you can solve. Most of this course is about that difference.

**Air-gapped and on-premises rule out SaaS entirely** — and not just the model.
OCR, parsing, embedding and reranking all need local alternatives too.

**Platforms make the easy 80% easy and the hard 20% impossible.** Custom
reranking, unusual document formats, domain-specific chunking. The hard 20% is
where the interesting work is.

**Data residency and pricing.** If customer data can't leave the country, or the
bill is in dollars and the revenue isn't, the technical comparison never happens.

## What's next

If you're still here, retrieval is worth building. Notebook 3 lays out what the
pieces are before you build any of them.